# TGEN-937-SR - Putative Gene Conversion Events Detected by Gubbins

# Import statements

In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns
#import pickle

%matplotlib inline

In [2]:
# https://bioframe.readthedocs.io/en/latest/guide-intervalops.html
import bioframe as bf


In [3]:
import json


In [4]:
# https://github.com/ipython/ipython/issues/10627
import os
os.environ['QT_QPA_PLATFORM']='offscreen'

import ete3 as ETE

from ete3 import Tree


### Import custom functions

In [5]:
#from gcutils.general import parse_PAFtools_VarTSV, label_DF_ByOvrLapGenes, label_PAF_DF_ByOvrLapGenes'

from gcutils.gubbinsfuncs import get_RecombEvents_From_Gubbins_GFF, parse_BaseReconstruction_EMBL_Gubbins

### Set matplotlib text export settings for Adobe Illustrator

In [6]:
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

#### Pandas Viewing Settings

In [7]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

# Import/parse processed H37rv genome annotations

In [8]:
RepoRef_Dir = "../../References"

AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir = f"{RepoRef_Dir}/201027_H37rv_AnnotatedGenes_And_IntergenicRegions"
H37Rv_GenomeAnnotations_Genes_TSV = f"{AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir}/H37Rv_GenomeAnnotations.Genes.tsv"

## H37Rv Gene Annotations TSV
H37Rv_GenomeAnno_Genes_DF = pd.read_csv(H37Rv_GenomeAnnotations_Genes_TSV, sep = "\t")
H37Rv_GeneInfo_Subset_DF = H37Rv_GenomeAnno_Genes_DF[["H37rv_GeneID", "Symbol", "Feature", "Functional_Category", "Is_Pseudogene", "Product", "PEandPPE_Subfamily", "ExcludedGroup_Category"]]

RvID_To_Symbol_Dict = dict(H37Rv_GeneInfo_Subset_DF[['H37rv_GeneID', 'Symbol']].values)

ESX_Genes_List_TSV = f"{RepoRef_Dir}/190927_H37rv_ListOf_ESXgenes.tsv"
Esx_Genes_DF = pd.read_csv(ESX_Genes_List_TSV, sep = '\t')

In [9]:
H37Rv_GenomeAnno_Genes_DF.head(1)

,Chrom,Start,End,Strand,H37rv_GeneID,Symbol,Feature,Functional_Category,Is_Pseudogene,Product,PEandPPE_Subfamily,ExcludedGroup_Category
0,NC_000962.3,0,1524,+,Rv0001,dnaA,CDS,information pathways,No,Chromosomal replication initiator protein DnaA,NaN,NotExcluded


# Parse in homology H37Rv mapping results (k19w19)

In [10]:
Main_Project_Dir = "/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V8"

H37_Rv_MM2_HomologyMapping_Dir = f"{Main_Project_Dir}/220502.H37Rv.HomologyMapping.k19w19.ProcessedData"       

# Define paths to output TSVs

### Homologous regions (MERGED)
H37Rv_HomologyRegions_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HomologousRegions.k19w19.tsv"

### Homology map (pairwise alignments)
H37Rv_HomologyMap_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HomologyMap.k19w19.tsv"
H37Rv_HomologyMap_NoOverlap_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HomologyMap.k19w19.NoOverlap.tsv"
H37Rv_HomologyMap_OnlyOverlap_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HomologyMap.k19w19.OnlyOverlap.tsv"

H37Rv_HomologyMap_NoOverlap_Processed_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HmMap.k19w19.NoOverlap.Processed.V2.tsv"

### Variants from the homology map alignments
H37Rv_HmMap_Var_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HomologyMap.k19w19.variants.tsv"
H37Rv_HmMap_Var_SNPs_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HomologyMap.k19w19.variants.snps.tsv"

### Parse in labeled homology-regions (w/ unique IDs)

HM_MergedRegions_Anno_DF = pd.read_csv(H37Rv_HomologyRegions_TSV, sep="\t")


### Parse in homology-map DFs (pairwise alignments between all homologous regions)

Mtb_HM_PAF_DF = pd.read_csv(H37Rv_HomologyMap_TSV, sep="\t")
Mtb_HM_PAF_DF_NoOverlapRegions = pd.read_csv(H37Rv_HomologyMap_NoOverlap_TSV, sep="\t")
Mtb_HM_PAF_DF_OnlyOverlapRegions = pd.read_csv(H37Rv_HomologyMap_OnlyOverlap_TSV, sep="\t")

HmPair_DF = pd.read_csv(H37Rv_HomologyMap_NoOverlap_Processed_TSV, sep="\t")

# Parse `TGEN-937-SR` sample metadata

In [11]:
Repo_MainDir = "../.."

Repo_DataDir = f"{Repo_MainDir}/Data"

Repo_RunInfoDir = f"{Repo_MainDir}/runInfo_TSVs"

TGen936SR_SampleInfo_CSV_PATH = f"{Repo_RunInfoDir}/TBportals.allTGEN_srrIds.forMax.csv"

TGen_1K_SM_V1_ResultsSummary_Dir = f"{Repo_DataDir}/Tgen1K_WGS_RunMetadata/211019_SM_TGen_1K_V1_ResultsSummary"

TGen936_SampleInfo_Filt_TSV_PATH = f"{Repo_DataDir}/Tgen1K_WGS_RunMetadata/211019_SM_TGen_1K_SampleInfo_V1.F2andCovFiltered.tsv"


TGenSR_WGS_Stats_Filt_DF = pd.read_csv(TGen936_SampleInfo_Filt_TSV_PATH, sep = "\t")

TGenSR_WGS_Stats_Filt_DF["PrimaryLineage"] = TGenSR_WGS_Stats_Filt_DF["PrimaryLineage_Ill"]
TGenSR_WGS_Stats_Filt_DF["SampleID"] = TGenSR_WGS_Stats_Filt_DF["SampleName"]
TGenSR_WGS_Stats_Filt_DF["Lineage"] = TGenSR_WGS_Stats_Filt_DF["LineageCall_Illumina"]
TGenSR_WGS_Stats_Filt_DF.shape

(937, 10)

### Define dictionaries that map sampleID to metadata labels

In [12]:
TGENSR_ID_To_PrimLineage_Dict = dict( TGenSR_WGS_Stats_Filt_DF[['SampleID', 'PrimaryLineage']].values)
TGENSR_ID_To_SubLineage_Dict  = dict( TGenSR_WGS_Stats_Filt_DF[["SampleID", "Lineage"]].values)
TGENSR_ID_To_Dataset_Dict     = dict( TGenSR_WGS_Stats_Filt_DF[['SampleID', 'Dataset_Tag']].values) 


### Define lists and dictionaries defininig which isolates were sequenced w/ PacBio HiFi for downstream verification

In [13]:

All_TBP_IDs_ForEventVerf = ['TB6599',
 'TB3305',
 'TB6976',
 'TB6596',
 'TB3898',
 'TB8073',
 'TB6733',
 'TB6846',
 'TB4414',
 'TB3706',
 'TB6778',
 'TB6973',
 'TB3572',
 'TB6977',
 'TB3256',
 'TB7340',
 'TB6765',
 'TB6755',
 'TB6807',
 'TB7044',
 'TB6552',
 'TB6786']

dictOf_TGEN_GCEventVerf_TBP_ID_Mappings  = {"Event_001" : {"Target" : "TB6733", "Control" : "TB3898"},
                                                 "Event_003" : {"Target" : "TB6599", "Control" : "TB6977"},
                                                 "Event_006" : {"Target" : "TB3305", "Control" : "TB3706"},
                                                 "Event_007" : {"Target" : "TB6755", "Control" : "TB7044"},
                                                 "Event_010" : {"Target" : "TB6552", "Control" : "TB6765"},
                                                 "Event_011" : {"Target" : "TB6778", "Control" : "TB6973"}, 
                                                 "Event_013" : {"Target" : "TB6786", "Control" : "TB3256"},
                                                 "Event_019" : {"Target" : "TB6977", "Control" : "TB6976"}, 
                                                 "Event_021" : {"Target" : "TB3572", "Control" : "TB6976"},
                                                 "Event_022" : {"Target" : "TB6596", "Control" : "TB8073"},    
                                                 "Event_024" : {"Target" : "TB7340", "Control" : "TB6807"},   
                                                 "Event_025" : {"Target" : "TB6846", "Control" : "TB4414"}}



# Convert to DataFrame
TGENSR_Events_ReseqIsolates_Info_DF = pd.DataFrame.from_dict(dictOf_TGEN_GCEventVerf_TBP_ID_Mappings, orient='index')
TGENSR_Events_ReseqIsolates_Info_DF = TGENSR_Events_ReseqIsolates_Info_DF.rename(columns={"Target": "Verification_IsolateID", "Control": "Control_IsolateID"})
TGENSR_Events_ReseqIsolates_Info_DF.index.name = "EventID"
TGENSR_Events_ReseqIsolates_Info_DF.reset_index(inplace=True)

print(TGENSR_Events_ReseqIsolates_Info_DF.shape)


# Create EventID → Target dictionary
EventID_to_TargetIsolateID = TGENSR_Events_ReseqIsolates_Info_DF.set_index("EventID")["Verification_IsolateID"].to_dict()

# Create EventID → Control dictionary
EventID_to_ControlIsolateID = TGENSR_Events_ReseqIsolates_Info_DF.set_index("EventID")["Control_IsolateID"].to_dict()


(12, 3)


In [14]:
TGENSR_Events_ReseqIsolates_Info_DF

,EventID,Verification_IsolateID,Control_IsolateID
0,Event_001,TB6733,TB3898
1,Event_003,TB6599,TB6977
2,Event_006,TB3305,TB3706
3,Event_007,TB6755,TB7044
4,Event_010,TB6552,TB6765
5,Event_011,TB6778,TB6973
6,Event_013,TB6786,TB3256
7,Event_019,TB6977,TB6976
8,Event_021,TB3572,TB6976
9,Event_022,TB6596,TB8073


In [15]:
len(All_TBP_IDs_ForEventVerf)

22

In [16]:
TGEN937SR_ReseqEventIDs = list( dictOf_TGEN_GCEventVerf_TBP_ID_Mappings.keys())
TGEN937SR_ReseqEventIDs

['Event_001',
 'Event_003',
 'Event_006',
 'Event_007',
 'Event_010',
 'Event_011',
 'Event_013',
 'Event_019',
 'Event_021',
 'Event_022',
 'Event_024',
 'Event_025']

In [17]:
len(TGEN937SR_ReseqEventIDs)

12

# Parse `Mtb151-MainAnalysis` Gubbins Results

### Define dictionary of file paths for Gubbins analysis

In [18]:
AnalysisName = "240923.WGA151CI.V8"

Main_Project_Dir = "/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V8"

Target_Output_Dir = f"{Main_Project_Dir}/{AnalysisName}"


WGA151_Gubbins_V1_OutputDir = f"{Target_Output_Dir}/Gubbins_v321_ExtSearch_SW_MS4_mpileup_SNVs_10AmbThresh"

WGA151_Gubbins_OutPrefix = "Gubbins"

WGA151_Gubbins_FullPrefix_PATH  = f"{WGA151_Gubbins_V1_OutputDir}/{WGA151_Gubbins_OutPrefix}"

WGA151_Gubbins_FilePath_Dict = {}

WGA151_Gubbins_FilePath_Dict["NodeLabelledTree_PATH"]        = f"{WGA151_Gubbins_FullPrefix_PATH}.node_labelled.final_tree.tre"
WGA151_Gubbins_FilePath_Dict["NodeToPriLineage_Dict_JSON"]   = f"{WGA151_Gubbins_FullPrefix_PATH}.NodeToPrimaryLineage.json"
WGA151_Gubbins_FilePath_Dict["BranchStats_CSV"]              = f"{WGA151_Gubbins_FullPrefix_PATH}.per_branch_statistics.csv"
WGA151_Gubbins_FilePath_Dict["BranchStats_WithLineage_CSV"]  = f"{WGA151_Gubbins_FullPrefix_PATH}.per_branch_statistics.WithLineagePerNode.tsv"
WGA151_Gubbins_FilePath_Dict["EventsPer_1kb_H37Rv_TSV"]      = f"{WGA151_Gubbins_FullPrefix_PATH}.H37Rv.EventsPer1kb.tsv"
WGA151_Gubbins_FilePath_Dict["EventsPer_Gene_H37Rv_TSV"]     = f"{WGA151_Gubbins_FullPrefix_PATH}.H37Rv.EventsPerGene.tsv"
WGA151_Gubbins_FilePath_Dict["EventsPer_HHR_H37Rv_TSV"]      = f"{WGA151_Gubbins_FullPrefix_PATH}.H37Rv.EventsPerMergedHomologousRegion.tsv"

WGA151_Gubbins_FilePath_Dict["RecombPred_Anno_TSV"]          = f"{WGA151_Gubbins_FullPrefix_PATH}.recombination_predictions.Anno.tsv"
WGA151_Gubbins_FilePath_Dict["BaseAncRec_All_TSV"]           = f"{WGA151_Gubbins_FullPrefix_PATH}.branch_base_reconstruction.AnnoByEvent.All.tsv"  
WGA151_Gubbins_FilePath_Dict["BaseAncRec_EventsOnly_TSV"]    = f"{WGA151_Gubbins_FullPrefix_PATH}.branch_base_reconstruction.AnnoByEvent.EventSNPsOnly.tsv" 




### Parse table of all putative recomb events detected in `WGA-151-CG` 

In [19]:
import ast


In [20]:
print( WGA151_Gubbins_FilePath_Dict["RecombPred_Anno_TSV"]) 

/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V8/240923.WGA151CI.V8/Gubbins_v321_ExtSearch_SW_MS4_mpileup_SNVs_10AmbThresh/Gubbins.recombination_predictions.Anno.tsv


In [21]:
# Parse annotated events TSV
WGA151_GRE_DF = pd.read_csv(WGA151_Gubbins_FilePath_Dict["RecombPred_Anno_TSV"], sep = "\t")

# Convert the string column to a list of strings
WGA151_GRE_DF['taxa_List'] = WGA151_GRE_DF['taxa_List'].apply(ast.literal_eval)

WGA151_GRE_DF["LenOfTaxaList"] = WGA151_GRE_DF["taxa_List"].apply(len)

WGA151_GRE_DF.shape

(324, 28)

In [22]:
WGA151_GRE_DF.head(1)

,seqname,source,feature,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,EventLen,HmAln_Count,HmAln_Ovrlap,HHR_Count,HHR_Ovrlap,LenOfTaxaList
0,NC_000962.3,GUBBINS,CDS,103600,104478,0.0,.,Node_148,Node_133,1737.432787,10,"[mada_2-31, mada_1-41, MT_0080, mada_102, TB33...",103599,104038.5,NaN,"Rv0093c,Rv0094c","Rv0093c,Rv0094c",False,False,True,False,Event_001,879,2,1,1,1,130


In [23]:
WGA151_GRE_DF.query("HHR_Count >= 2 ").shape

(1, 28)

In [24]:
WGA151_GRE_DF.query("HHR_Count >= 2 ")

,seqname,source,feature,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,EventLen,HmAln_Count,HmAln_Ovrlap,HHR_Count,HHR_Ovrlap,LenOfTaxaList
219,NC_000962.3,GUBBINS,CDS,2866607,2867756,0.0,.,Node_132,Node_131,1451.162781,10,"[R21893, R30420, R32929, R26778, R23146, R2898...",2866606,2867181.0,lineage2,"lppA,lppB","Rv2543,Rv2544",False,False,False,True,Event_220,1150,2,1,2,1,61


### Parse Gubbins recombination events counted over "1 kb windows", "Gene-level"

In [25]:
WGA151_GRE_GeneLevel_DF = pd.read_csv(WGA151_Gubbins_FilePath_Dict["EventsPer_Gene_H37Rv_TSV"], sep = "\t")

WGA151_GRE_GeneLevel_Atleast1_DF = WGA151_GRE_GeneLevel_DF.query("pGCE_Count > 0")
WGA151_GRE_GeneLevel_Atleast1_DF.shape

(76, 14)

In [26]:
WGA151_GRE_PerHHRStats_DF = pd.read_csv(WGA151_Gubbins_FilePath_Dict["EventsPer_HHR_H37Rv_TSV"] , sep = "\t")
WGA151_GRE_PerHHRStats_DF.shape

(197, 19)

In [27]:
WGA151_GRE_PerHHRStats_DF.query("pGCE_Count > 0").shape

(54, 19)

In [28]:
WGA151_GRE_PerHHRStats_DF["pGCE_Count"].sum()

296

In [29]:
WGA151_GRE_PerHHRStats_DF.query("pGCE_Count > 0").shape

(54, 19)

In [30]:
WGA151_GRE_PerHHRStats_DF.head(1)

,HmRegion_Num,Chr,Start,End,Num_Ovrlap_Hm_Regions,Center,Length,num_HomologRegions_NonOvrlap,num_HomologRegions_NonOvrlap_MinSeqID99,num_HomologRegions_NonOvrlap_MinSeqID100,Overlap_Genes,Overlap_TE,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,HmRegionID,pGCE_Count,CenterOfRegion
0,0,NC_000962.3,80184,80523,1,80353.5,339,1,1,1,Rv0071,0,False,False,False,True,HmRegion_000,0,80353.5


### C) Parse in WGA-151CI Gene Conversion Counts per High-Homology-Region

In [31]:
RecombEvent_To_HmRegion_CompDir = f"{Target_Output_Dir}/RecombEvent-To-HmRegion-Comparison-V2"

HmPairs_MappedEvents_TSV = f"{RecombEvent_To_HmRegion_CompDir}/240718.HmPair.MappedEventCounts.tsv"
HmRegions_MappedEvents_TSV = f"{RecombEvent_To_HmRegion_CompDir}/240718.HmRegions.MappedEventCounts.tsv"

WGA151_HHRs_GCEs_DonorAndEventCt_DF = pd.read_csv(HmRegions_MappedEvents_TSV, sep = "\t")
WGA151_HHRs_GCEs_DonorAndEventCt_DF["N_Events_Mapped"] = WGA151_HHRs_GCEs_DonorAndEventCt_DF["N_Events_HighQC"] 
WGA151_HHRs_GCEs_DonorAndEventCt_DF.shape

(197, 23)

In [32]:
WGA151_HHRs_GCEs_DonorAndEventCt_DF.head()  

,HmRegion_Num,Chr,Start,End,Center,Length,num_HomologRegions_NonOvrlap_MinSeqID99,num_HomologRegions_NonOvrlap_MinSeqID100,Overlap_Genes,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,HmRegionID,Norm_Donor_Count,N_Events_HighQC,N_Events_Putative,EventSkew,DonAccCt,EventSkew_Norm,FractionMapped,PR_SetID,N_Events_Mapped
0,0,NC_000962.3,80184,80523,80353.5,339,1,1,Rv0071,False,False,False,True,HmRegion_000,0.0,0,0,0.0,0.0,NaN,NaN,PR_Set_1,0
1,1,NC_000962.3,80623,82664,81643.5,2041,1,1,"Rv0072,Rv0073",False,False,False,True,HmRegion_001,0.0,0,0,0.0,0.0,NaN,NaN,PR_Set_2,0
2,2,NC_000962.3,103705,105130,104417.5,1425,2,2,"Rv0094c,Rv0095c",False,False,True,False,HmRegion_002,10.5,19,31,8.5,29.5,0.288136,0.612903,PR_Set_3,19
3,3,NC_000962.3,149571,149808,149689.5,237,1,1,PE_PGRS2,False,True,False,False,HmRegion_003,0.0,0,0,0.0,0.0,NaN,NaN,PR_Set_4,0
4,4,NC_000962.3,177203,177447,177325.0,244,1,1,NaN,False,False,False,True,HmRegion_004,0.0,0,0,0.0,0.0,NaN,NaN,PR_Set_5,0


In [33]:
WGA151_HHRs_GCEs_DonorAndEventCt_DF["N_Events_Putative"].sum()

296

In [34]:
WGA151_HHRs_GCEs_DonorAndEventCt_DF["N_Events_HighQC"].sum()

213

In [35]:
WGA151_HHRs_GCEs_DonorAndEventCt_DF["Length"].sum()

257094

In [36]:
257094 / 4411532

0.05827771395515209

# Parse `TGEN-937-SR` Gubbins Results

### Define all useful Gubbins results file paths

In [37]:
# Define varaint calling pipeline output directories

TGen1K_SRWGS_OutputDir = "/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb-GeneConv/250721.TGEN1K.GCAnalysis"

# Define path to TGEN936K Gubbins output (SR-WGS based predictions)
TGen1K_SRWGS_Gubbins_OutDir = f"{TGen1K_SRWGS_OutputDir}/Gubbins_Analysis_V1"

Gubbins_V1_OutputDir = TGen1K_SRWGS_Gubbins_OutDir

TGENSR_Gubbins_OutPrefix = "Tgen_937CI.Gubbins.FromSNVs.V1"

TGENSR_Gubbins_FullPrefix_PATH  = f"{TGen1K_SRWGS_Gubbins_OutDir}/{TGENSR_Gubbins_OutPrefix}"

TGENSR_Gubbins_FilePath_Dict = {}

TGENSR_Gubbins_FilePath_Dict["NodeLabelledTree_PATH"]        = f"{TGENSR_Gubbins_FullPrefix_PATH}.node_labelled.final_tree.tre"
TGENSR_Gubbins_FilePath_Dict["NodeToPriLineage_Dict_JSON"]   = f"{TGENSR_Gubbins_FullPrefix_PATH}.NodeToPrimaryLineage.json"
TGENSR_Gubbins_FilePath_Dict["BranchStats_CSV"]              = f"{TGENSR_Gubbins_FullPrefix_PATH}.per_branch_statistics.csv"
TGENSR_Gubbins_FilePath_Dict["BranchStats_WithLineage_CSV"]  = f"{TGENSR_Gubbins_FullPrefix_PATH}.per_branch_statistics.WithLineagePerNode.tsv"
TGENSR_Gubbins_FilePath_Dict["EventsPer_1kb_H37Rv_TSV"]      = f"{TGENSR_Gubbins_FullPrefix_PATH}.H37Rv.EventsPer1kb.tsv"
TGENSR_Gubbins_FilePath_Dict["EventsPer_Gene_H37Rv_TSV"]     = f"{TGENSR_Gubbins_FullPrefix_PATH}.H37Rv.EventsPerGene.tsv"
TGENSR_Gubbins_FilePath_Dict["EventsPer_HHR_H37Rv_TSV"]      = f"{TGENSR_Gubbins_FullPrefix_PATH}.H37Rv.EventsPerMergedHomologousRegion.tsv"

TGENSR_Gubbins_FilePath_Dict["RecombPred_Anno_TSV"]          = f"{TGENSR_Gubbins_FullPrefix_PATH}.recombination_predictions.Anno.tsv"
#TGENSR_Gubbins_FilePath_Dict["RecombPred_GFF"]              = f"{TGENSR_Gubbins_FullPrefix_PATH}.recombination_predictions.gff"
#TGENSR_Gubbins_FilePath_Dict["RecombPred_RenamedContig_GFF"] = f"{TGENSR_Gubbins_FullPrefix_PATH}.recombination_predictions.RenamedCHR.gff"
#TGENSR_Gubbins_FilePath_Dict["RecombPred_RenamedContig_BED"] = f"{TGENSR_Gubbins_FullPrefix_PATH}.recombination_predictions.RenamedCHR.bed"
TGENSR_Gubbins_FilePath_Dict["BaseAncRec_All_TSV"]           = f"{TGENSR_Gubbins_FullPrefix_PATH}.branch_base_reconstruction.AnnoByEvent.All.tsv"  
TGENSR_Gubbins_FilePath_Dict["BaseAncRec_EventsOnly_TSV"]    = f"{TGENSR_Gubbins_FullPrefix_PATH}.branch_base_reconstruction.AnnoByEvent.EventSNPsOnly.tsv" 


# TGENSR_Gubbins_FilePath_Dict["_____________________"] = f"{TGENSR_Gubbins_FullPrefix_PATH}.___________"


### Parse table of all putative recomb events detected in `TGEN-936-SR` 

In [38]:
import ast


In [39]:
print( TGENSR_Gubbins_FilePath_Dict["RecombPred_Anno_TSV"]) 

/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb-GeneConv/250721.TGEN1K.GCAnalysis/Gubbins_Analysis_V1/Tgen_937CI.Gubbins.FromSNVs.V1.recombination_predictions.Anno.tsv


In [40]:
# Parse annotated events TSV
TGENSR_GRE_DF = pd.read_csv(TGENSR_Gubbins_FilePath_Dict["RecombPred_Anno_TSV"], sep = "\t")

# Convert the string column to a list of strings
TGENSR_GRE_DF['taxa_List'] = TGENSR_GRE_DF['taxa_List'].apply(ast.literal_eval)

TGENSR_GRE_DF["LenOfTaxaList"] = TGENSR_GRE_DF["taxa_List"].apply(len)

TGENSR_GRE_DF.shape

(27, 29)

In [41]:
TGENSR_GRE_DF.head(1)

,seqname,source,feature,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,HmAln_Count,HmAln_Ovrlap,HHR_Count,HHR_Ovrlap,Overlap_HHRs,LenOfTaxaList
0,NC_000962.3,GUBBINS,CDS,333120,333212,0.0,.,internal_985,internal_986,158.013591,8,"[SRR10379945, SRR7516364, SRR10380193, SRR1037...",333119,333165.5,93,lineage4,"vapC25,vapB25","Rv0277c,Rv0277A",False,False,False,True,Event_001,1,1,1,1,HmRegion_009,34


In [42]:
TGENSR_GRE_DF.query("HHR_Count >= 2 ").shape

(2, 29)

In [43]:
TGENSR_GRE_DF.shape

(27, 29)

In [44]:
TGENSR_GRE_DF.query(" HHR_Ovrlap == 0 ")


,seqname,source,feature,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,HmAln_Count,HmAln_Ovrlap,HHR_Count,HHR_Ovrlap,Overlap_HHRs,LenOfTaxaList
20,NC_000962.3,GUBBINS,CDS,3477266,3477370,0.0,.,internal_1082,SRR10397163,175.415705,5,[SRR10397163],3477265,3477317.5,105,lineage4,Rv3108,Rv3108,False,False,False,True,Event_021,0,0,0,0,NaN,1


### Parse Gubbins recombination events counted over "Gene-level", "High-Homology Regions"

In [45]:
TGENSR_GRE_GeneLevel_DF = pd.read_csv(TGENSR_Gubbins_FilePath_Dict["EventsPer_Gene_H37Rv_TSV"], sep = "\t")
TGENSR_GRE_GeneLevel_Atleast1_DF = TGENSR_GRE_GeneLevel_DF.query("pGCE_Count > 0")
TGENSR_GRE_GeneLevel_Atleast1_DF.shape

(25, 14)

In [46]:
TGENSR_GRE_PerHHRStats_DF = pd.read_csv(TGENSR_Gubbins_FilePath_Dict["EventsPer_HHR_H37Rv_TSV"] , sep = "\t")
TGENSR_GRE_PerHHRStats_DF.shape

(197, 20)

In [47]:
TGENSR_GRE_PerHHRStats_DF.query("pGCE_Count > 0").shape

(17, 20)

In [48]:
TGENSR_GRE_PerHHRStats_DF["pGCE_Count"].sum()

28

In [49]:
TGENSR_GRE_PerHHRStats_DF.query("pGCE_Count > 0").shape

(17, 20)

In [50]:
TGENSR_GRE_PerHHRStats_DF.head(1)

,HmRegion_Num,Chr,Start,End,Num_Ovrlap_Hm_Regions,Center,Length,num_HomologRegions_NonOvrlap,num_HomologRegions_NonOvrlap_MinSeqID99,num_HomologRegions_NonOvrlap_MinSeqID100,Overlap_Genes,Overlap_TE,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,HmRegionID,pGCE_Count,CenterOfRegion,Overlap_GC_EventIDs
0,0,NC_000962.3,80184,80523,1,80353.5,339,1,1,1,Rv0071,0,False,False,False,True,HmRegion_000,0,80353.5,NaN


In [51]:
#TGENSR_GRE_PerHHRStats_DF[["HmRegionID", "Overlap_Genes", "Start", "End",  "Length", "pGCE_Count", "num_HomologRegions_NonOvrlap"]].query("pGCE_Count > 0")

In [52]:
TGENSR_GRE_DF

,seqname,source,feature,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,HmAln_Count,HmAln_Ovrlap,HHR_Count,HHR_Ovrlap,Overlap_HHRs,LenOfTaxaList
0,NC_000962.3,GUBBINS,CDS,333120,333212,0.0,.,internal_985,internal_986,158.013591,8,"[SRR10379945, SRR7516364, SRR10380193, SRR1037...",333119,333165.5,93,lineage4,"vapC25,vapB25","Rv0277c,Rv0277A",False,False,False,True,Event_001,1,1,1,1,HmRegion_009,34
1,NC_000962.3,GUBBINS,CDS,473800,473971,0.0,.,internal_1262,SRR6807674,70.598206,6,[SRR6807674],473799,473885.0,172,lineage4,Rv0393,Rv0393,False,False,True,False,Event_002,1,1,1,1,HmRegion_018,1
2,NC_000962.3,GUBBINS,CDS,841087,841448,0.0,.,internal_1061,internal_1062,1023.196321,14,"[SRR10380227, SRR10379958]",841086,841267.0,362,lineage4,"vapB31,vapC31","Rv0748,Rv0749",False,False,False,True,Event_003,1,1,1,1,HmRegion_031,2
3,NC_000962.3,GUBBINS,CDS,842026,842065,0.0,.,internal_941,SRR6807728,1309.669305,9,[SRR6807728],842025,842045.0,40,lineage4,Rv0750,Rv0750,False,False,False,True,Event_004,1,1,1,1,HmRegion_032,1
4,NC_000962.3,GUBBINS,CDS,842051,842111,0.0,.,internal_1177,internal_1181,374.835261,7,"[SRR6807675, SRR10379983, SRR6807700, SRR68076...",842050,842080.5,61,lineage4,Rv0750,Rv0750,False,False,False,True,Event_005,1,1,1,1,HmRegion_032,145
5,NC_000962.3,GUBBINS,CDS,1094538,1095317,0.0,.,internal_1505,SRR10397175,92.819244,7,[SRR10397175],1094537,1094927.0,780,lineage2,"Rv0979c,rpmF,PE_PGRS18","Rv0979c,Rv0979A,Rv0980c",False,True,False,False,Event_006,3,1,2,1,"HmRegion_039,HmRegion_040",1
6,NC_000962.3,GUBBINS,CDS,1276321,1276588,0.0,.,internal_943,internal_946,69.803852,5,"[SRR10380134, SRR10380230, SRR10379994, SRR103...",1276320,1276454.0,268,lineage4,Rv1148c,Rv1148c,False,False,True,False,Event_007,1,1,1,1,HmRegion_049,21
7,NC_000962.3,GUBBINS,CDS,1339399,1339905,0.0,.,internal_1166,internal_1167,600.958317,5,"[SRR6807683, SRR10380192, SRR7516429, SRR68077...",1339398,1339651.5,507,lineage4,PPE18,Rv1196,False,True,False,False,Event_008,2,1,1,1,HmRegion_052,159
8,NC_000962.3,GUBBINS,CDS,1339894,1340208,0.0,.,internal_941,SRR6807728,1282.018277,8,[SRR6807728],1339893,1340050.5,315,lineage4,PPE18,Rv1196,False,True,False,False,Event_009,2,1,1,1,HmRegion_052,1
9,NC_000962.3,GUBBINS,CDS,1340052,1341254,0.0,.,internal_953,SRR10380108,397.130081,5,[SRR10380108],1340051,1340652.5,1203,lineage4,"PPE18,esxK,esxL","Rv1196,Rv1197,Rv1198",True,True,False,False,Event_010,6,1,2,1,"HmRegion_052,HmRegion_053",1


# Part 1: Look over ALL events (`N=27`) detected using SR-WGS with the `TGEN-937` dataset

In [53]:
TargetColn = ["EventID","Overlap_Genes",  "Lineage", "HHR_Count", "HHR_Ovrlap", "Parent_Node", "Child_Node", "start_1based", "end_1based", "snp_count", "LenOfTaxaList", 'taxa_List',]   


In [54]:
TGENSR_GRE_DF[TargetColn]

,EventID,Overlap_Genes,Lineage,HHR_Count,HHR_Ovrlap,Parent_Node,Child_Node,start_1based,end_1based,snp_count,LenOfTaxaList,taxa_List
0,Event_001,"vapC25,vapB25",lineage4,1,1,internal_985,internal_986,333120,333212,8,34,"[SRR10379945, SRR7516364, SRR10380193, SRR1037..."
1,Event_002,Rv0393,lineage4,1,1,internal_1262,SRR6807674,473800,473971,6,1,[SRR6807674]
2,Event_003,"vapB31,vapC31",lineage4,1,1,internal_1061,internal_1062,841087,841448,14,2,"[SRR10380227, SRR10379958]"
3,Event_004,Rv0750,lineage4,1,1,internal_941,SRR6807728,842026,842065,9,1,[SRR6807728]
4,Event_005,Rv0750,lineage4,1,1,internal_1177,internal_1181,842051,842111,7,145,"[SRR6807675, SRR10379983, SRR6807700, SRR68076..."
5,Event_006,"Rv0979c,rpmF,PE_PGRS18",lineage2,2,1,internal_1505,SRR10397175,1094538,1095317,7,1,[SRR10397175]
6,Event_007,Rv1148c,lineage4,1,1,internal_943,internal_946,1276321,1276588,5,21,"[SRR10380134, SRR10380230, SRR10379994, SRR103..."
7,Event_008,PPE18,lineage4,1,1,internal_1166,internal_1167,1339399,1339905,5,159,"[SRR6807683, SRR10380192, SRR7516429, SRR68077..."
8,Event_009,PPE18,lineage4,1,1,internal_941,SRR6807728,1339894,1340208,8,1,[SRR6807728]
9,Event_010,"PPE18,esxK,esxL",lineage4,2,1,internal_953,SRR10380108,1340052,1341254,5,1,[SRR10380108]


#### QC and exploration of specific isolates

In [55]:
TGENSR_GRE_DF[ TGENSR_GRE_DF["taxa_List"].str.join(",").str.contains("SRR10380218") ]

,seqname,source,feature,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,HmAln_Count,HmAln_Ovrlap,HHR_Count,HHR_Ovrlap,Overlap_HHRs,LenOfTaxaList
18,NC_000962.3,GUBBINS,CDS,3377271,3377320,0.0,.,internal_1060,internal_1061,613.252749,8,"[SRR10380227, SRR10379958, SRR10380218, SRR103...",3377270,3377295.0,50,lineage4,PPE46,Rv3018c,False,True,False,False,Event_019,1,1,1,1,HmRegion_152,6


In [56]:
TGENSR_GRE_DF[ TGENSR_GRE_DF["taxa_List"].str.join(",").str.contains("SRR10379962") ]

,seqname,source,feature,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,HmAln_Count,HmAln_Ovrlap,HHR_Count,HHR_Ovrlap,Overlap_HHRs,LenOfTaxaList
6,NC_000962.3,GUBBINS,CDS,1276321,1276588,0.0,.,internal_943,internal_946,69.803852,5,"[SRR10380134, SRR10380230, SRR10379994, SRR103...",1276320,1276454.0,268,lineage4,Rv1148c,Rv1148c,False,False,True,False,Event_007,1,1,1,1,HmRegion_049,21


In [57]:
TGENSR_GRE_DF[ TGENSR_GRE_DF["taxa_List"].str.join(",").str.contains("SRR10379958") ]

,seqname,source,feature,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,HmAln_Count,HmAln_Ovrlap,HHR_Count,HHR_Ovrlap,Overlap_HHRs,LenOfTaxaList
2,NC_000962.3,GUBBINS,CDS,841087,841448,0.0,.,internal_1061,internal_1062,1023.196321,14,"[SRR10380227, SRR10379958]",841086,841267.0,362,lineage4,"vapB31,vapC31","Rv0748,Rv0749",False,False,False,True,Event_003,1,1,1,1,HmRegion_031,2
18,NC_000962.3,GUBBINS,CDS,3377271,3377320,0.0,.,internal_1060,internal_1061,613.252749,8,"[SRR10380227, SRR10379958, SRR10380218, SRR103...",3377270,3377295.0,50,lineage4,PPE46,Rv3018c,False,True,False,False,Event_019,1,1,1,1,HmRegion_152,6


In [58]:
TGENSR_GRE_DF.query("EventID == 'Event_019'")["taxa_List"].values

array([list(['SRR10380227', 'SRR10379958', 'SRR10380218', 'SRR10380217', 'SRR10397107', 'SRR6356960'])],
      dtype=object)

##### NOTE, SRR10380218 (`TB6977`/`DNA0530`) can be used as a WITHIN EVENT VERFICATION ISOLATE FOR EVENT_019

## Define the HHRs which have a MAPPED and PUTATIVE GCEs in WGA-151 dataset

In [59]:
WGA151_HHRs_GCEs_DonorAndEventCt_DF.shape

(197, 23)

In [60]:
WGA151_mGCE_HHRs_DF = WGA151_HHRs_GCEs_DonorAndEventCt_DF.query("N_Events_Mapped > 0")
WGA151_mGCE_HHRs_DF.shape

(45, 23)

In [61]:
WGA151_pGCE_HHRs_DF = WGA151_HHRs_GCEs_DonorAndEventCt_DF.query("N_Events_Putative > 0")
WGA151_pGCE_HHRs_DF.shape

(54, 23)

In [62]:
WGA151_mGCE_HHRs_DF.shape

(45, 23)

In [63]:
TGENSR_GRE_DF.shape

(27, 29)

### Annotate all `TGEN937SR` detected GC events by overlap with a HHRs w/ GCE in primary analysis (`WGA151`)

In [64]:
RE_CoordCols = ("seqname", "start_0based", "end_1based")
HmRegion_CoordCols = ("Chr", "Start", "End")

TGENSR_GRE_AnnoByWGA151Ovrlap_DF = bf.count_overlaps(TGENSR_GRE_DF,
                                                     WGA151_pGCE_HHRs_DF,
                                                     cols1 = RE_CoordCols,
                                                     cols2 = HmRegion_CoordCols).rename(columns={'count': 'Ovrlap_HRR_Wi_pGCE_InWGA151'})
 

TGENSR_GRE_AnnoByWGA151Ovrlap_DF = bf.count_overlaps(TGENSR_GRE_AnnoByWGA151Ovrlap_DF,
                                                     WGA151_mGCE_HHRs_DF,
                                                     cols1 = RE_CoordCols,
                                                     cols2 = HmRegion_CoordCols).rename(columns={'count': 'Ovrlap_HRR_Wi_mGCE_InWGA151'})



TGENSR_GRE_AnnoByWGA151Ovrlap_DF = bf.count_overlaps(TGENSR_GRE_AnnoByWGA151Ovrlap_DF,
                                                     WGA151_GRE_DF,
                                                     cols1 = RE_CoordCols,
                                                     cols2 = RE_CoordCols).rename(columns={'count': 'Ovrlap_Any_pGCE_InWGA151'})
 



TGENSR_GRE_AnnoByWGA151Ovrlap_DF["Reseq_With_PacBioHifi"] = TGENSR_GRE_AnnoByWGA151Ovrlap_DF["EventID"].isin(TGEN937SR_ReseqEventIDs)


TGENSR_GRE_AnnoByWGA151Ovrlap_DF["Reseq_VerficationIsolate"] = TGENSR_GRE_AnnoByWGA151Ovrlap_DF["EventID"].map(EventID_to_TargetIsolateID).fillna("")
TGENSR_GRE_AnnoByWGA151Ovrlap_DF["Reseq_ControlIsolate"] = TGENSR_GRE_AnnoByWGA151Ovrlap_DF["EventID"].map(EventID_to_ControlIsolateID).fillna("")


## Look at overlap between TGEN-SR detected events and HHRs w/ gene conversion events in the primary analysis

#### Specific Questions:
1) Across ALL TGEN-SR events detected, how many overlap with any HHR?
2) Across ALL TGEN-SR events detected, how many overlap with a HHR with a **putative** GC event in the WGA-151CI analysis (primary analysis)?
2) Across ALL TGEN-SR events detected, how many overlap with a HHR with a **mapped** GC event in the WGA-151CI analysis (primary analysis)?


#### How many total TGEN-SR events? (27 events detected)

In [65]:
TGENSR_GRE_DF.shape

(27, 29)

In [66]:
TGENSR_GRE_DF.head(1)[TargetColn]

,EventID,Overlap_Genes,Lineage,HHR_Count,HHR_Ovrlap,Parent_Node,Child_Node,start_1based,end_1based,snp_count,LenOfTaxaList,taxa_List
0,Event_001,"vapC25,vapB25",lineage4,1,1,internal_985,internal_986,333120,333212,8,34,"[SRR10379945, SRR7516364, SRR10380193, SRR1037..."


#### How many TGEN-SR events overlap with a HHR? (26/27 events)

In [67]:
TGENSR_GRE_DF.query("HHR_Count >= 1").shape

(26, 29)

#### How many TGEN-SR events DO NOT overlap with a HHR? (1/27 events)

In [68]:
TGENSR_GRE_DF.query("HHR_Count == 0").shape

(1, 29)

In [69]:
TGENSR_GRE_DF.query("HHR_Count == 0")[TargetColn]

,EventID,Overlap_Genes,Lineage,HHR_Count,HHR_Ovrlap,Parent_Node,Child_Node,start_1based,end_1based,snp_count,LenOfTaxaList,taxa_List
20,Event_021,Rv3108,lineage4,0,0,internal_1082,SRR10397163,3477266,3477370,5,1,[SRR10397163]


### How many `TGENSR` events overlap with any HH region with a putative GC event in `WGA151`?

In [70]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF.query("Ovrlap_HRR_Wi_pGCE_InWGA151 >= 1").shape

(24, 35)

### How many `TGENSR` events overlap with any HH region with a MAPPED GC event in `WGA151`?

In [71]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF.query("Ovrlap_HRR_Wi_mGCE_InWGA151 >= 1").shape

(21, 35)

### How many `TGENSR` events DO NOT overlap with any HH region with a putative GC event in `WGA151`? (3 Events, 2 in HHRs but last one has no HHR overlapping)

In [72]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF.query("Ovrlap_HRR_Wi_pGCE_InWGA151 == 0").shape

(3, 35)

In [73]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF.query("Ovrlap_HRR_Wi_pGCE_InWGA151 == 0")

,seqname,source,feature,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,HmAln_Count,HmAln_Ovrlap,HHR_Count,HHR_Ovrlap,Overlap_HHRs,LenOfTaxaList,Ovrlap_HRR_Wi_pGCE_InWGA151,Ovrlap_HRR_Wi_mGCE_InWGA151,Ovrlap_Any_pGCE_InWGA151,Reseq_With_PacBioHifi,Reseq_VerficationIsolate,Reseq_ControlIsolate
14,NC_000962.3,GUBBINS,CDS,2358862,2358880,0.0,.,internal_1065,SRR10397107,431.784393,4,[SRR10397107],2358861,2358870.5,19,lineage4,Rv2100,Rv2100,False,False,True,False,Event_015,1,1,1,1,HmRegion_102,1,0,0,0,False,,
15,NC_000962.3,GUBBINS,CDS,2975496,2975617,0.0,.,internal_1054,SRR10397246,383.566781,5,[SRR10397246],2975495,2975556.0,122,lineage4,Rv2651c,Rv2651c,False,False,False,True,Event_016,1,1,1,1,HmRegion_128,1,0,0,0,False,,
20,NC_000962.3,GUBBINS,CDS,3477266,3477370,0.0,.,internal_1082,SRR10397163,175.415705,5,[SRR10397163],3477265,3477317.5,105,lineage4,Rv3108,Rv3108,False,False,False,True,Event_021,0,0,0,0,NaN,1,0,0,0,True,TB3572,TB6976


### How many `TGENSR` events DO NOT overlap with any HH region ? (1 Event in `Rv3108` gene)

In [74]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF.query("HHR_Count == 0").shape

(1, 35)

In [75]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF.query("HHR_Count == 0")

,seqname,source,feature,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,HmAln_Count,HmAln_Ovrlap,HHR_Count,HHR_Ovrlap,Overlap_HHRs,LenOfTaxaList,Ovrlap_HRR_Wi_pGCE_InWGA151,Ovrlap_HRR_Wi_mGCE_InWGA151,Ovrlap_Any_pGCE_InWGA151,Reseq_With_PacBioHifi,Reseq_VerficationIsolate,Reseq_ControlIsolate
20,NC_000962.3,GUBBINS,CDS,3477266,3477370,0.0,.,internal_1082,SRR10397163,175.415705,5,[SRR10397163],3477265,3477317.5,105,lineage4,Rv3108,Rv3108,False,False,False,True,Event_021,0,0,0,0,NaN,1,0,0,0,True,TB3572,TB6976


## Look at overall import states for all TGENSR detected events (N = 27)

In [76]:
TargetColn_Set2 = ["EventID", "Overlap_Genes", "snp_count", "LenOfTaxaList",
                   "Child_Node",
                   "Lineage",   
                   "HHR_Ovrlap",
                   "Ovrlap_HRR_Wi_pGCE_InWGA151",
                   "Reseq_With_PacBioHifi",
                   "Reseq_VerficationIsolate", "Reseq_ControlIsolate"] 


In [77]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF[TargetColn_Set2]

,EventID,Overlap_Genes,snp_count,LenOfTaxaList,Child_Node,Lineage,HHR_Ovrlap,Ovrlap_HRR_Wi_pGCE_InWGA151,Reseq_With_PacBioHifi,Reseq_VerficationIsolate,Reseq_ControlIsolate
0,Event_001,"vapC25,vapB25",8,34,internal_986,lineage4,1,1,True,TB6733,TB3898
1,Event_002,Rv0393,6,1,SRR6807674,lineage4,1,1,False,,
2,Event_003,"vapB31,vapC31",14,2,internal_1062,lineage4,1,1,True,TB6599,TB6977
3,Event_004,Rv0750,9,1,SRR6807728,lineage4,1,1,False,,
4,Event_005,Rv0750,7,145,internal_1181,lineage4,1,1,False,,
5,Event_006,"Rv0979c,rpmF,PE_PGRS18",7,1,SRR10397175,lineage2,1,2,True,TB3305,TB3706
6,Event_007,Rv1148c,5,21,internal_946,lineage4,1,1,True,TB6755,TB7044
7,Event_008,PPE18,5,159,internal_1167,lineage4,1,1,False,,
8,Event_009,PPE18,8,1,SRR6807728,lineage4,1,1,False,,
9,Event_010,"PPE18,esxK,esxL",5,1,SRR10380108,lineage4,1,2,True,TB6552,TB6765


## Look at overall import states for resequenced TGENSR detected events (N = 12)

In [78]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF[TargetColn_Set2].query("Reseq_With_PacBioHifi == True")

,EventID,Overlap_Genes,snp_count,LenOfTaxaList,Child_Node,Lineage,HHR_Ovrlap,Ovrlap_HRR_Wi_pGCE_InWGA151,Reseq_With_PacBioHifi,Reseq_VerficationIsolate,Reseq_ControlIsolate
0,Event_001,"vapC25,vapB25",8,34,internal_986,lineage4,1,1,True,TB6733,TB3898
2,Event_003,"vapB31,vapC31",14,2,internal_1062,lineage4,1,1,True,TB6599,TB6977
5,Event_006,"Rv0979c,rpmF,PE_PGRS18",7,1,SRR10397175,lineage2,1,2,True,TB3305,TB3706
6,Event_007,Rv1148c,5,21,internal_946,lineage4,1,1,True,TB6755,TB7044
9,Event_010,"PPE18,esxK,esxL",5,1,SRR10380108,lineage4,1,2,True,TB6552,TB6765
10,Event_011,PPE18,7,3,internal_1140,lineage4,1,1,True,TB6778,TB6973
12,Event_013,PPE19,6,4,internal_1039,lineage4,1,1,True,TB6786,TB3256
18,Event_019,PPE46,8,6,internal_1061,lineage4,1,1,True,TB6977,TB6976
20,Event_021,Rv3108,5,1,SRR10397163,lineage4,0,0,True,TB3572,TB6976
21,Event_022,PPE56,7,4,internal_1023,lineage4,1,1,True,TB6596,TB8073


In [79]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF[TargetColn_Set2].query("Reseq_With_PacBioHifi == True").shape

(12, 11)

# Save TSV of `TGENSR` event info annotated by overlap w/ HHRs and WGA151 events

In [80]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF.shape

(27, 35)

In [81]:
TGENSR_GubbinsEvents_AnnoByOvrlapWiWGA151Events_TSV = f"{TGENSR_Gubbins_FullPrefix_PATH}.recombination_predictions.AnnoByOvrlapWiPrimAnalysis.V1.tsv"

TGENSR_GRE_AnnoByWGA151Ovrlap_DF.to_csv(TGENSR_GubbinsEvents_AnnoByOvrlapWiWGA151Events_TSV, sep = "\t", index=False)


In [82]:
!wc -l $TGENSR_GubbinsEvents_AnnoByOvrlapWiWGA151Events_TSV

28 /n/data1/hms/dbmi/farhat/mm774/Projects/Mtb-GeneConv/250721.TGEN1K.GCAnalysis/Gubbins_Analysis_V1/Tgen_937CI.Gubbins.FromSNVs.V1.recombination_predictions.AnnoByOvrlapWiPrimAnalysis.V1.tsv


# Test reading in TSV of 27 events from `TGENSR` annotated by overlap w/ primary analysis

In [83]:
TGENSR_GubbinsEvents_AnnoByOvrlapWiWGA151Events_TSV = f"{TGENSR_Gubbins_FullPrefix_PATH}.recombination_predictions.AnnoByOvrlapWiPrimAnalysis.V1.tsv"

TGENSR_GRE_AnnoByWGA151Ovrlap_DF = pd.read_csv(TGENSR_GubbinsEvents_AnnoByOvrlapWiWGA151Events_TSV, sep = "\t")
TGENSR_GRE_AnnoByWGA151Ovrlap_DF.shape

(27, 35)

In [84]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF.head()

,seqname,source,feature,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,HmAln_Count,HmAln_Ovrlap,HHR_Count,HHR_Ovrlap,Overlap_HHRs,LenOfTaxaList,Ovrlap_HRR_Wi_pGCE_InWGA151,Ovrlap_HRR_Wi_mGCE_InWGA151,Ovrlap_Any_pGCE_InWGA151,Reseq_With_PacBioHifi,Reseq_VerficationIsolate,Reseq_ControlIsolate
0,NC_000962.3,GUBBINS,CDS,333120,333212,0.0,.,internal_985,internal_986,158.013591,8,"['SRR10379945', 'SRR7516364', 'SRR10380193', '...",333119,333165.5,93,lineage4,"vapC25,vapB25","Rv0277c,Rv0277A",False,False,False,True,Event_001,1,1,1,1,HmRegion_009,34,1,1,0,True,TB6733,TB3898
1,NC_000962.3,GUBBINS,CDS,473800,473971,0.0,.,internal_1262,SRR6807674,70.598206,6,['SRR6807674'],473799,473885.0,172,lineage4,Rv0393,Rv0393,False,False,True,False,Event_002,1,1,1,1,HmRegion_018,1,1,1,2,False,NaN,NaN
2,NC_000962.3,GUBBINS,CDS,841087,841448,0.0,.,internal_1061,internal_1062,1023.196321,14,"['SRR10380227', 'SRR10379958']",841086,841267.0,362,lineage4,"vapB31,vapC31","Rv0748,Rv0749",False,False,False,True,Event_003,1,1,1,1,HmRegion_031,2,1,1,2,True,TB6599,TB6977
3,NC_000962.3,GUBBINS,CDS,842026,842065,0.0,.,internal_941,SRR6807728,1309.669305,9,['SRR6807728'],842025,842045.0,40,lineage4,Rv0750,Rv0750,False,False,False,True,Event_004,1,1,1,1,HmRegion_032,1,1,1,3,False,NaN,NaN
4,NC_000962.3,GUBBINS,CDS,842051,842111,0.0,.,internal_1177,internal_1181,374.835261,7,"['SRR6807675', 'SRR10379983', 'SRR6807700', 'S...",842050,842080.5,61,lineage4,Rv0750,Rv0750,False,False,False,True,Event_005,1,1,1,1,HmRegion_032,145,1,1,3,False,NaN,NaN
